In [ ]:
import json, os, time, uuid
from pathlib import Path
import requests

BASE_URL = os.getenv('ECHO_BASE_URL', 'http://127.0.0.1:8002').rstrip('/')
VLLM_BASE_URL = os.getenv('GEMMA4_VLLM_BASE_URL', 'http://127.0.0.1:8003/v1').rstrip('/')
MODEL = os.getenv('GEMMA4_BASE_MODEL', 'gemma4_e2b')
DEMO_SEED_TOKEN = os.getenv('ECHO_DEMO_SEED_TOKEN', 'kaggle-demo-seed')
TOKEN = os.getenv('ECHO_TOKEN', '')
USER_ID = ''

def show(title, value):
    print('\n' + '=' * 80)
    print(title)
    print('=' * 80)
    if isinstance(value, (dict, list)):
        print(json.dumps(value, indent=2)[:5000])
    else:
        print(value)

def headers():
    return {'Authorization': f'Bearer {TOKEN}'} if TOKEN else {}

def echo(method, path, payload=None, params=None, timeout=180):
    print(f'Echo {method} {path} ...', flush=True)
    r = requests.request(method, BASE_URL + path, headers=headers(), json=payload, params=params, timeout=timeout)
    print(f'-> HTTP {r.status_code}', flush=True)
    r.raise_for_status()
    return r.json() if r.text else {}

print('Config loaded')
print('BASE_URL =', BASE_URL)
print('VLLM_BASE_URL =', VLLM_BASE_URL)
print('MODEL =', MODEL)


In [ ]:
print('Checking Echo and vLLM health...', flush=True)
echo_health = requests.get(BASE_URL + '/health', timeout=10).json()
models = requests.get(VLLM_BASE_URL + '/models', timeout=10).json()
show('Echo health', echo_health)
show('vLLM models', models)
assert any(m.get('id') == MODEL for m in models.get('data', [])), f'{MODEL} not found in vLLM models'
print('Health checks passed')


In [ ]:
global TOKEN, USER_ID
print('Seeding public-safe demo user...', flush=True)
seed = requests.post(
    BASE_URL + '/v1/demo/seed',
    headers={'x-echo-demo-token': DEMO_SEED_TOKEN},
    json={'scenario': 'proof_camera_maya', 'reset': False, 'stable': False},
    timeout=180,
)
print('seed HTTP', seed.status_code, flush=True)
seed.raise_for_status()
seed_json = seed.json()
TOKEN = seed_json['token']
USER_ID = seed_json['user']['id']
show('Seed result', {'user_id': USER_ID, 'result': seed_json.get('result')})
show('Authenticated user', echo('GET', '/auth/me'))


In [ ]:
nonce = uuid.uuid4().hex[:10]
print('Calling direct vLLM with nonce', nonce, flush=True)
direct = requests.post(
    VLLM_BASE_URL + '/chat/completions',
    json={
        'model': MODEL,
        'messages': [{'role': 'user', 'content': f'Reply in one sentence and include nonce {nonce}.'}],
        'temperature': 0.2,
        'max_tokens': 80,
    },
    timeout=180,
)
direct.raise_for_status()
direct_json = direct.json()
text = direct_json['choices'][0]['message']['content']
show('Direct live Gemma response', {'id': direct_json.get('id'), 'model': direct_json.get('model'), 'text': text})
assert nonce in text, 'Gemma response did not include the nonce'


In [ ]:
nonce = uuid.uuid4().hex[:10]
chat = echo('POST', '/v1/chat/completions', {
    'model': MODEL,
    'messages': [{'role': 'user', 'content': f'This is a live Echo chat test. Mention nonce {nonce} and one proof step.'}],
    'temperature': 0.3,
    'max_tokens': 180,
})
content = chat['choices'][0]['message']['content']
show('Echo live chat', {'id': chat.get('id'), 'model': chat.get('model'), 'content': content})
print('Echo chat returned a live completion. Nonce requested:', nonce)


In [ ]:
artifact = {
    'artifact_type': 'mobile_camera_payload',
    'scene': 'Garden sensor prototype beside field-test notes.',
    'visible_text': [
        'Garden sensor v2',
        'Outdoor test stable for 40 minutes',
        'Cost reduced from $18 to $11',
        'Teacher says Maya explains electronics clearly',
    ],
    'user_caption': 'This worked outside without internet.',
    'goal': 'Win a scholarship or apprenticeship by showing real technical proof.',
    'opportunity_type': 'scholarship',
}
vision = echo('POST', '/v1/vision/analyze', artifact, timeout=240)
show('Proof Camera / vision analyze', vision)


In [ ]:
trace = echo('GET', '/v1/training/pipeline-trace', params={'lane': 'gemma4_e2b', 'prepare': '1', 'write': '1'}, timeout=300)
show('Shadow Clone pipeline trace', {
    'ready': trace.get('ready'),
    'prep': trace.get('prep'),
    'raw_counts': trace.get('raw_counts'),
    'datasets': {k: {'rows': v.get('rows'), 'ready': v.get('ready'), 'path': v.get('path')} for k, v in (trace.get('datasets') or {}).items()},
})


In [ ]:
print('Triggering bounded real demo training now. This may restart vLLM.', flush=True)
evidence = echo('POST', '/v1/training/demo-loop', {'lane': 'gemma4_e2b', 'max_pairs': 8, 'max_steps': 8, 'min_pairs': 4}, timeout=1800)
show('Bounded demo training evidence', {
    'status': evidence.get('status'),
    'profile': evidence.get('profile'),
    'real_training': evidence.get('real_training'),
    'bounds': evidence.get('bounds'),
    'dataset': evidence.get('dataset'),
    'runtime_steps': evidence.get('runtime_steps'),
    'promotion': evidence.get('promotion'),
    'before_after': evidence.get('before_after'),
})
